# Task Notebook — Args, Kwargs & Unpacking: The Final Boss

This is the final practical notebook of Section 03. Across seven tasks you will master calling styles, variadic packing, keyword-only rules, and call-site unpacking, finishing with a full smart-bill function that combines every idea.

**How to use it.** Each task has a description, a `TODO` cell, a collapsible hint, and a worked solution. Try first, compare after. The setup cell loads `products_catalog.csv` and prepares the name-to-price dictionary the later tasks use.

**What you will practise.**
1. Positional and keyword calls, including the duplicate-value error.
2. `*args` packing, including the empty case.
3. `**kwargs` packing and graceful `.get` handling.
4. Call-site star openers `f(*list)` and `f(**dict)`.
5. Keyword-only parameters with a bare star.
6. A real cart-total helper over catalogue prices.
7. The boss: a smart bill maker with discount, GST, member bonus, unknown flags, invalid-item skipping, and both call styles.

**Key idea.** The star is symmetric: it packs at the definition and unpacks at the call. Master both directions and flexible APIs become straightforward. This notebook also closes Section 03, so the final checklist is a quick review of everything you have learned.

## 1. Setup — Loading the Catalogue

The setup reuses the catalogue loader and builds `name2price`, a dictionary mapping each product name to its price. Every later task that needs a price reads from this dictionary.

The loader tries two file paths and raises `FileNotFoundError` if neither works. With the lookup table ready, the tasks can focus purely on signatures and unpacking rather than file handling.

In [1]:
# ⚙️ SETUP — catalog loader (18-mode!)
import csv

def load_products():
    for path in ["../../00_Resources/datasets/products_catalog.csv",
                 "00_Python/00_Resources/datasets/products_catalog.csv"]:
        try:
            with open(path, newline="", encoding="utf-8") as f:
                return list(csv.DictReader(f))
        except FileNotFoundError:
            continue
    raise FileNotFoundError("products_catalog.csv nahi mila!")

_products = load_products()
name2price = {p["product_name"]: float(p["price"]) for p in _products}
print(f"Catalog ready: {len(name2price)} products — signature ring set! 🎪")
# Expected OUTPUT:
# Catalog ready: 30 products — signature ring set! 🎪

Catalog ready: 30 products — signature ring set! 🎪


## 2. Task 1 — Call-Style Wrestling

Goal: define `show(a, b=100)`, then exercise four calls: one positional, two positional, all-keyword, and finally the invalid `show(5, a=2)` inside a `try/except`.

The last call supplies `a` both positionally and by keyword, so Python raises a got-multiple-values error. The task makes the matching rules concrete — positionals fill parameters first, and a keyword cannot rebind one already filled.

## ⭐ Task 1: Call-Style Wrestling 🤼

Create `def show(a, b=100)` that prints `f"a={a} b={b}"`. Calls:
1. `show(1)` 2. `show(1, 2)` 3. `show(b=5, a=7)` 4. `show(5, a=2)` — the 🚨 ones in try/except!

💡 **Hint:** in the 4th call 'a' gets double-values — the got-multiple-values TypeError!

In [2]:
# TODO 1-4

<details><summary>✅ Solution</summary>

a=1 b=100 | a=1 b=2 | a=7 b=5 | show() got multiple values for argument 'a' 💥
</details>

In [3]:
# ✅ SOLUTION — Task 1
def show(a, b=100):
    print(f"a={a} b={b}")

show(1)
show(1, 2)
show(b=5, a=7)
try:
    show(5, a=2)
except TypeError as e:
    print("dupe-arg 🚨:", e)
# Expected OUTPUT:
# a=1 b=100
# a=1 b=2
# a=7 b=5
# dupe-arg 🚨: show() got multiple values for argument 'a'

a=1 b=100
a=1 b=2
a=7 b=5
dupe-arg 🚨: show() got multiple values for argument 'a'


## 3. Task 2 — The Chemist's Bag: *args

Goal: write `sum_all(*nums)` that returns the sum of any number of values, including none, and `show_type(*items)` that prints the type and value it received.

The point is that `args` is always a tuple. `sum_all()` returns 0 because `sum(())` is 0, and `show_type('apple', 'mango', 'kiwi')` prints a tuple. This is variadic packing at its simplest.

## ⭐ Task 2: Chemist's Bag — *args 🛍️

1. Write `def sum_all(*nums): return sum(nums)`
2. `sum_all(1, 2, 3)` and `sum_all()` — print both
3. `def show_type(*items): print(type(items).__name__, items)` — call it with 3 fruits

💡 **Hint:** print the TUPLE identity of args — do not forget this one!


In [4]:
# TODO 1-3

<details><summary>✅ Solution</summary>

6 / 0 (empty safe nil!) | tuple ('apple', 'mango', 'kiwi') — args-pack proven! 🛍️
</details>

In [5]:
# ✅ SOLUTION — Task 2
def sum_all(*nums):
    return sum(nums)

print("sum_all(1,2,3):", sum_all(1, 2, 3))
print("sum_all()     :", sum_all())

def show_type(*items):
    print("args type:", type(items).__name__, "| value:", items)

show_type("apple", "mango", "kiwi")
# Expected OUTPUT:
# sum_all(1,2,3): 6
# sum_all()     : 0
# args type: tuple | value: ('apple', 'mango', 'kiwi')

sum_all(1,2,3): 6
sum_all()     : 0
args type: tuple | value: ('apple', 'mango', 'kiwi')


## 4. Task 3 — Opening the Gift-Bag: **kwargs

Goal: write `user_card(**info)` that prints `name` and `city` using `.get(..., 'unknown')`, call it with an extra `age` keyword, and also call it with no arguments.

The extra `age` is accepted but ignored, demonstrating the open bag; the empty call prints `unknown` for both fields, showing graceful defaults. This is how optional configuration is handled safely.

## ⭐⭐ Task 3: Gift-Bag Open — **kwargs 🎁

1. `def user_card(**info):` — print `name=` and `city=` with `.get(..., "unknown")`
2. Call with `name="Sana", city="Delhi", age=21` (IGNORE age inside!)
3. One EMPTY call — it should print unknown-unknown

💡 **Hint:** `.get("key", "unknown")` — the graceful brother of missing keys!

In [6]:
# TODO 1-3

<details><summary>✅ Solution</summary>

Sana/Delhi (age silently ignored!), empty → unknown/unknown. Gift-bag handled! 🎁
</details>

In [7]:
# ✅ SOLUTION — Task 3
def user_card(**info):
    print(f"name={info.get('name', 'unknown')} city={info.get('city', 'unknown')}")

user_card(name="Sana", city="Delhi", age=21)
user_card()
# Expected OUTPUT:
# name=Sana city=Delhi
# name=unknown city=unknown

name=Sana city=Delhi
name=unknown city=unknown


## 5. Task 4 — The Star-Opener Trio

Goal: use call-site unpacking three ways: pass a list into `f(a, b, c)` with `*`, pass a dictionary into `greet(name, city)` with `**`, and call `max(*scores)` on a list.

This confirms the call-side direction of the star: `*` spreads positionals and `**` spreads keywords, and both can feed any function that accepts unpacked arguments, including built-ins like `max`.

## ⭐⭐ Task 4: Star-Opener Trio ⭐

1. `def f(a, b, c): return a * b + c` — call with `vals = [2, 3, 4]` (open with the star!)
2. `d = {"name": "Ravi", "city": "Jaipur"}` — call `def greet(name, city)` with `**`
3. `max([4, 9, 2, 8])` without slicing — the star-unpack style!

💡 **Hint:** max itself takes *args: `max(*list)` — a direct entry!

In [8]:
# TODO 1-3

<details><summary>✅ Solution</summary>

2*3+4 = 10; "Hello Ravi from Jaipur!"; max(*[...]) = 9. Star openers done! ⭐
</details>

In [9]:
# ✅ SOLUTION — Task 4
def f(a, b, c):
    return a * b + c

vals = [2, 3, 4]
print("f(*vals)  :", f(*vals))

def greet(name, city):
    print(f"Hello {name} from {city}!")

d = {"name": "Ravi", "city": "Jaipur"}
greet(**d)

scores = [4, 9, 2, 8]
print("max(*list):", max(*scores))
# Expected OUTPUT:
# f(*vals)  : 10
# Hello Ravi from Jaipur!
# max(*list): 9

f(*vals)  : 10
Hello Ravi from Jaipur!
max(*list): 9


## 6. Task 5 — The Keyword-Only Zone

Goal: define `pizza(size, *, extra_cheese=False, garlic_bread=False)`, call it with defaults, call it with one keyword flag, and attempt to pass the flag positionally inside a `try/except`.

The bare star makes the flags keyword-only, so the positional attempt raises `TypeError`. This enforces explicit, readable call sites and is a common convention in public APIs.

## ⭐⭐⭐ Task 5: Kw-Only Zone 👮

Create `def pizza(size, *, extra_cheese=False, garlic_bread=False)` that prints it all:
1. `pizza("large")` 2. `pizza("medium", extra_cheese=True)`
3. `pizza("small", True)` — 🚨 in try/except! (kw-only does NOT accept positional!)

💡 **Hint:** the args after `*` are STRICT keywords-only — they do not go without their name!

In [10]:
# TODO 1-3

<details><summary>✅ Solution</summary>

Defaults False-False ✅; explicit kw ✅; positional-for-kw-only REJECTED 👮!
</details>

In [11]:
# ✅ SOLUTION — Task 5
def pizza(size, *, extra_cheese=False, garlic_bread=False):
    print(f"pizza={size} cheese={extra_cheese} garlic={garlic_bread}")

pizza("large")
pizza("medium", extra_cheese=True)
try:
    pizza("small", True)
except TypeError as e:
    print("kw-only 🚨:", e)
# Expected OUTPUT:
# pizza=large cheese=False garlic=False
# pizza=medium cheese=True garlic=False
# kw-only 🚨: pizza() takes 1 positional argument but 2 were given

pizza=large cheese=False garlic=False
pizza=medium cheese=True garlic=False
kw-only 🚨: pizza() takes 1 positional argument but 2 were given


## 7. Task 6 — Mixed-Cart Calculator

Goal: write `cart_total(*item_names, rounding=True)` that sums catalogue prices for the given names and optionally rounds the result to two decimals. Then call it with two items and with two cheap items using `rounding=False`.

This combines `*args` with a keyword-only option and a generator expression for the sum, mirroring how a real cart helper would look.

## ⭐⭐⭐ Task 6 (💾): Mixed-Cart Calculator 🛒

Use the ready setup `name2price`:
1. Create `def cart_total(*item_names, rounding=True)`:
   - sum the prices of the given items (dict lookups!)
   - if `rounding=True`, `round()` to 2 decimals and return, otherwise raw
2. Print `cart_total("Laptop", "Headphones")` (₹50,000 expected!)
3. Print `cart_total("Bread", "Milk_1L", rounding=False)` (₹105!)

💡 **Hint:** `sum(name2price[i] for i in item_names)` — a genexpr will come in handy!

In [12]:
# TODO 1-3

<details><summary>✅ Solution</summary>

Laptop+Headphones = ₹50,000.0; Bread+Milk_1L = ₹105.0 (if raw, zero precision — that's it!).
Cart machine ready! 🛒
</details>

In [13]:
# ✅ SOLUTION — Task 6
def cart_total(*item_names, rounding=True):
    total = sum(name2price[i] for i in item_names)
    return round(total, 2) if rounding else total

print("cart 1:", cart_total("Laptop", "Headphones"))
print("cart 2:", cart_total("Bread", "Milk_1L", rounding=False))
# Expected OUTPUT:
# cart 1: 50000.0
# cart 2: 105.0

cart 1: 50000.0
cart 2: 105.0


## 8. Task 7 — The Smart Bill Maker (Boss)

Goal: build `smart_bill(cashier, *items, discount=0, gst=0.0, delivery=0, member=False, **flags)` that prints a formatted bill. It must sum valid items, skip invalid ones with a warning, add a 5% bonus discount for members, apply GST, print unknown flags, and support both keyword calls and unpacked `*`/`**` calls.

This is the synthesis of the entire topic: every parameter kind, both call styles, and real data. Completing it closes Section 03's final notebook.

## 🏆 Task 7 (💾 BOSS): SMART BILL MAKER — Full-Signature Champion! 🧾👑

Chhota-Mart needs *One* function that handles everything:
`smart_bill(cashier, *items, discount=0, gst=0.0, delivery=0, member=False, **flags)`

What to do:
1. Subtotal = sum the prices of the items (name2price lookups, an invalid item → a "⚠️ skipped" line!)
2. If member, add +5% EXTRA to the discount (`discount + 0.05 if member`)
3. Pretty print — the EXACT format:
```
  🧾 CHHOTA-MART BILL             cashier: Rinku
  ──────────────────────────────────────────
    - Laptop            ₹48,000
    - Smartphone        ₹15,000
    - Headphones        ₹ 2,000
  ──────────────────────────────────────────
  subtotal ₹65,000 | member+disc 15% → ₹55,250
  +5% GST → ₹58,012.5 | +delivery ₹0
   FINAL: ₹58,012.5 | flags: ['parking'] | THANKS! 🙏
```
4. Call-1: `smart_bill("Rinku", "Laptop", "Smartphone", "Headphones", discount=0.10, gst=0.05, member=True, parking="free")`
5. Call-2 (unpacking!): `bill_config = {"discount": 0.05, "gst": 0.05}` + `smart_bill("Mohan", *["Bread", "Milk_1L"], **bill_config)` → ₹105 → -5% → 99.75 → +5% GST = 104.74 (round 2)

💡 **Hints:**
- member-discount: `d_total = discount + (0.05 if member else 0)`
- PRICE column right-align: the `{price:>8,.0f}` style; match the item gaps with a name-width of 18!
- Call-2 rounding: stage-wise `round(x, 2)` — will come out as ₹104.74

**Challenge:** Gracefully skip invalid items — print the "⚠️ skipped: Mogambo" line!
(section-03's FINAL BOSS! 🏆)

In [14]:
# TODO 1-5

<details><summary>✅ Solution — CHAMPION CASHIER!</summary>

Bill-1: ₹65,000 subtotal → 15% member-discount → ₹55,250 → +5% GST = ₹58,012.5 FINAL!
Bill-2 (unpacking): ₹105 → ₹99.75 → ₹104.74 FINAL (star-openers flexed!)
Invalid items gracefully skipped. Section-03 trophy UNLOCKED! 🏆🎉
</details>

In [15]:
# ✅ SOLUTION — Task 7 (Smart Bill Maker 🧾👑)
def smart_bill(cashier, *items, discount=0, gst=0.0, delivery=0, member=False, **flags):
    print(f"  🧾 CHHOTA-MART BILL             cashier: {cashier}")
    print("  " + "─" * 44)
    subtotal = 0
    for i in items:
        if i in name2price:
            print(f"    - {i:<18} ₹{name2price[i]:>8,.0f}")
            subtotal += name2price[i]
        else:
            print(f"    ⚠️ skipped: {i}")
    print("  " + "─" * 44)
    d = discount + (0.05 if member else 0)
    after_d = round(subtotal * (1 - d), 2)
    after_gst = round(after_d * (1 + gst), 2)
    final = round(after_gst + delivery, 2)
    print(f"  subtotal ₹{subtotal:,.0f} | member+disc {int(d*100)}% → ₹{after_d:,.0f}")
    print(f"  +{int(gst*100)}% GST → ₹{after_gst:,.1f} | +delivery ₹{delivery:,.0f}")
    print(f"   FINAL: ₹{final:,.1f} | flags: {sorted(flags)} | THANKS! 🙏")

# Call-1: full kwargs fire!
smart_bill("Rinku", "Laptop", "Smartphone", "Headphones",
           discount=0.10, gst=0.05, member=True, parking="free")
print()
# Call-2: star/double-star UNPACKING flex! ⭐🎁
bill_config = {"discount": 0.05, "gst": 0.05}
smart_bill("Mohan", *["Bread", "Milk_1L", "Mogambo"], **bill_config)
# Expected OUTPUT:
#   🧾 CHHOTA-MART BILL             cashier: Rinku
#   ──────────────────────────────────────────
#     - Laptop             ₹  48,000
#     - Smartphone         ₹  15,000
#     - Headphones         ₹   2,000
#   ──────────────────────────────────────────
#   subtotal ₹65,000 | member+disc 15% → ₹55,250
#   +5% GST → ₹58,012.5 | +delivery ₹0
#    FINAL: ₹58,012.5 | flags: ['parking'] | THANKS! 🙏
#
#   🧾 CHHOTA-MART BILL             cashier: Mohan
#   ──────────────────────────────────────────
#     - Bread              ₹      45
#     - Milk_1L            ₹      60
#     ⚠️ skipped: Mogambo
#   ──────────────────────────────────────────
#   subtotal ₹105 | member+disc 5% → ₹100
#   +5% GST → ₹104.7 | +delivery ₹0
#    FINAL: ₹104.7 | flags: [] | THANKS! 🙏

  🧾 CHHOTA-MART BILL             cashier: Rinku
  ────────────────────────────────────────────
    - Laptop             ₹  48,000
    - Smartphone         ₹  15,000
    - Headphones         ₹   2,000
  ────────────────────────────────────────────
  subtotal ₹65,000 | member+disc 15% → ₹55,250
  +5% GST → ₹58,012.5 | +delivery ₹0
   FINAL: ₹58,012.5 | flags: ['parking'] | THANKS! 🙏

  🧾 CHHOTA-MART BILL             cashier: Mohan
  ────────────────────────────────────────────
    - Bread              ₹      45
    - Milk_1L            ₹      60
    ⚠️ skipped: Mogambo
  ────────────────────────────────────────────
  subtotal ₹105 | member+disc 5% → ₹100
  +5% GST → ₹104.7 | +delivery ₹0
   FINAL: ₹104.7 | flags: [] | THANKS! 🙏


## Summary

- Positional arguments match by order, keyword arguments by name, and duplicates raise `TypeError`.
- `*args` packs any number of positionals into a tuple, including the empty tuple.
- `**kwargs` packs named extras into a dictionary and pairs with `.get` for safe defaults.
- Call-site stars do the opposite: `f(*list)` and `f(**dict)` expand stored data into arguments.
- A bare `*` creates a keyword-only zone that rejects positional calls.
- The boss task combined all of these into a smart bill that handles discounts, GST, member bonuses, unknown flags, and invalid items.

Section 03 is complete: you can now reason about iteration, lazy pipelines, copying, mutability, and flexible signatures.